# Walkthrough: a solver through the typed `Task` ADT

Traces one prompt end-to-end through `DispatchTurnstyle` — the production consumer of
`turnstyle.dispatch`. We use the richest path, a **snarks** `MultipleChoice` solver,
because a single pass touches every stage:

```
prompt
  │  parse(prompt, ctx)            # smart-constructor cascade -> a Task variant
  ▼
MultipleChoice(options, gather=None, selection=None)
  │  enrich(task, prompt, ctx)     # model forward pass fills typed fields
  ▼
MultipleChoice(..., selection=Deliberated(margin))
  │  solve(task, prompt, ctx)      # total match -> _solve_choice -> ChoiceProbe (L13 argmax)
  ▼
Answer(text="(B)", source="choice_probe")
  │  SequenceLogitsProcessor       # bias the model to *emit* the answer
  ▼
grounded output: "(B)"
```

Run with the turnstyle venv kernel. The probe-fit cell takes ~1–2 min (autoprobe sweep);
everything else is fast.

## Setup

In [ ]:
import os, sys, json
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
sys.path.insert(0, "/Users/jdonaldson/Projects/turnstyle/src")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
device = "mps" if torch.backends.mps.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL)
mdl = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).to(device).eval()
print("model on", device)

## The prompt
A sarcasm multiple-choice question — option (B) is the sarcastic one.

In [ ]:
BBH = "/Users/jdonaldson/Projects/swollm/data/bbh_cache"
snarks = json.load(open(f"{BBH}/snarks.json"))
prompt = snarks[0]["input"]
print(prompt)
print("\ngold:", snarks[0]["target"])

## Stage 1 — `parse`: raw prompt → a typed `Task`

`parse` is a cascade of **smart constructors** in priority order: each tries to claim the
prompt and returns `None` if it can't. The first that succeeds wins. For snarks, the value
solvers decline (arithmetic is **coverage-gated** — see below), and the option markers
`(A)/(B)` produce a `MultipleChoice`.

In [ ]:
from turnstyle.dispatch import parse, Ctx

ctx = Ctx(model=mdl, tokenizer=tok, device=device)
task = parse(prompt, ctx)
print("parse ->", task)

In [ ]:
# Why didn't arithmetic grab the numbers in the prose? The coverage gate.
from turnstyle.arithmetic import parse_expression
from turnstyle.dispatch import _covers

res = parse_expression(prompt)
print("arithmetic sub-expression match:", res)
if res:
    print("covers the prompt? ", _covers(res[0], prompt), "  (a value solver must explain >= 50% of the prompt)")

## Stage 2 — `enrich`: model-based detectors fill the typed fields

`parse` is cheap (structural). `enrich` runs the one model forward pass: `detect_selection_shape`
appends `"\nAnswer: ("`, logit-lenses every layer to find the **decision layer**, and returns
`PriorLocked` (model committed early, ~L2–3, bias-driven) or `Deliberated(margin)` (committed
late). This is `option_decision_layer.py` productized.

In [ ]:
from turnstyle.dispatch import enrich

task = enrich(task, prompt, ctx)
print("enriched ->", task)
print("selection shape:", task.selection)

## Stage 3 — fit the probe (one-time), then `solve`

`MultipleChoice` is the one variant that needs a model probe. `fit_choice` runs `autoprobe`
over the task's examples and attaches the fitted per-option `ProbeArtifact`. **~1–2 min.**

In [ ]:
from turnstyle import DispatchTurnstyle

dt = DispatchTurnstyle(mdl, tok, device)
result = dt.fit_choice(snarks)          # autoprobe sweep: layer x finder x mode, 5-fold CV
print(result.reason)

`solve` is the total `match`. It lands on `MultipleChoice -> _solve_choice`, which fires the
**`ChoiceProbe`**: it scores `P(sarcastic)` at **L13** for each option's last token and takes the
**argmax**. This is the "interrupt argmax and solve" finding — read stage-1 per-option scoring
(which the model does well) and do the comparison externally, skipping the model's weak stage-2
selection (the knockout proved that's causal).

In [ ]:
from turnstyle.dispatch import solve

ans = solve(task, prompt, dt.ctx)       # dt.ctx now carries the fitted artifact
print(ans)

## Stage 4 — grounding: the model is *made* to say it

Back in `Turnstyle.generate`, the `Answer` becomes a `SequenceLogitsProcessor` that biases the
model to emit `Answer.text`. The deterministic answer the probe computed becomes the model's
actual output — that's the neurosymbolic loop.

In [ ]:
text, proof = dt.generate(prompt, max_new_tokens=8)
print("grounded output:", repr(text), " | gold:", snarks[0]["target"])

## Contrast — the skeleton without the model in the middle

A **deterministic** solver is the same pipeline with stages 2–3 collapsed: `parse` already knows
the answer, no `enrich`, no probe. An **abstain** (a knowledge prompt with no variant) routes to
`FreeForm -> Abstain`, and the base class falls back to plain generation.

In [ ]:
# Deterministic: answer known at parse; no forward pass until grounding.
t, _ = dt.generate("What is 3 * (4 + 5)?", max_new_tokens=6)
print("arithmetic   ->", repr(t))

# Abstain: no variant claims it -> FreeForm -> Abstain -> plain generation.
from turnstyle.dispatch import run, Abstain
r = run("Who composed the obscure opera nobody remembers?", ctx)
print("knowledge    ->", type(r).__name__, "| reason:", getattr(r, "reason", "-"))

## This one path is the whole project arc

- **Stage 1 coverage gate** — "commitment = coverage", the principle that replaced ad-hoc guards.
- **Stage 2 `detect_selection_shape`** — the decision-layer experiment, now a runtime detector.
- **Stage 3 `ChoiceProbe` @ L13 + external argmax** — "interrupt argmax and solve"; reads the
  model's good per-option scoring, skips its weak selection. Causally confirmed by the attention
  knockout.
- **Stage 4 `SequenceLogitsProcessor`** — neurosymbolic grounding.
- **Throughout** — `Task` sum type + `solve -> Answer | Abstain`, exhaustiveness compiler-checked.

Memories: `mc_selection_two_stage`, `typed_adt_dispatch_pattern`, `commitment_coverage_routing`.